In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import netCDF4 as nc
from scipy.interpolate import griddata


ModuleNotFoundError: No module named 'numpy'

In [ ]:
fname = f'/srv/scratch/z3533156/26year_BRAN2020/outer_avg_01461.nc'

dataset = nc.Dataset(fname)

lon_rho = np.transpose(dataset.variables['lon_rho'], axes=(1, 0))
lat_rho = np.transpose(dataset.variables['lat_rho'], axes=(1, 0))
mask_rho = np.transpose(dataset.variables['mask_rho'], axes=(1, 0))
h = np.transpose(dataset.variables['h'], axes=(1, 0))
# f = np.transpose(dataset.variables['f'], axes=(1, 0))
angle = dataset.variables['angle'][0, 0]
z_r = np.load('/srv/scratch/z5297792/z_r.npy')
z_r = np.transpose(z_r, (1, 2, 0))

def distance(lat1, lon1, lat2, lon2):
    EARTH_RADIUS = 6357
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return EARTH_RADIUS * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

j_mid, i_mid = lon_rho.shape[1] // 2, lon_rho.shape[0] // 2

dx = distance(lat_rho[:-1, j_mid], lon_rho[:-1, j_mid],
              lat_rho[1:, j_mid], lon_rho[1:, j_mid])
dy = distance(lat_rho[i_mid, :-1], lon_rho[i_mid, :-1],
              lat_rho[i_mid, 1:], lon_rho[i_mid, 1:])

x_grid = np.insert(np.cumsum(dx), 0, 0)
y_grid = np.insert(np.cumsum(dy), 0, 0)
X_grid, Y_grid = np.meshgrid(x_grid, y_grid, indexing='ij')


In [ ]:
df_ameda = pd.read_pickle(
    '/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset/AMEDA/df_AMEDA.pkl'
).rename(columns={
    'x1': 'lon',
    'y1': 'lat',
    'step_datetime': 'Date',
    'eddy_id': 'Eddy',
    'velmax1': 'velmax',
    'vort1': 'w',
    'OW1': 'OW',
    'rmax1': 'rmax',
    'ke1': 'ke',
    'aire1': 'area',
    'deta1': 'deta'
})

# ensure numerical values (not objects)
num_cols = [
    'lon', 'lat', 'Eddy', 'type',
    'velmax', 'w', 'OW', 'rmax',
    'ke', 'area', 'deta'
]
for col in num_cols:
    if col in df_ameda.columns:
        df_ameda[col] = np.asarray(df_ameda[col].to_list(), dtype='float64')


points = np.column_stack([lon_rho.ravel(), lat_rho.ravel()])
values = np.column_stack([X_grid.ravel(), Y_grid.ravel()])

df_ameda[['xc', 'yc']] = griddata(
    points,
    values,
    (df_ameda['lon'].to_numpy(), df_ameda['lat'].to_numpy()),
    method='linear'
)

df_ameda['Cyc'] = np.where(df_ameda['type'].eq(1.0), 'CE', 'AE')

df_ameda['Date'] = pd.to_datetime(df_ameda['Date']) + pd.Timedelta(days=1)

df_ameda['Day'] = (
    df_ameda['Date'] - pd.Timestamp('1990-01-01')
).dt.days

df_ameda['fname'] = df_ameda['Day'].map(
    lambda day: f"/srv/scratch/z3533156/26year_BRAN2020/outer_avg_{1461 + ((day - 1462) // 30) * 30:05}.nc"
)

df_ameda['Age'] = df_ameda.groupby('Eddy')['Eddy'].transform('size')

df_ameda['Eddy'] = df_ameda['Eddy'].rank(method='dense').astype(int)

df_ameda['ic'] = np.abs(df_ameda['xc'].to_numpy()[:, None] - x_grid).argmin(axis=1)
df_ameda['jc'] = np.abs(df_ameda['yc'].to_numpy()[:, None] - y_grid).argmin(axis=1)

df_ameda['w'] = -1 * df_ameda['w'] # SH

cols = [
    'Eddy', 'Day', 'Cyc', 'lon', 'lat',
    'ic', 'jc', 'xc', 'yc',
    'w', 'OW', 'ke', 'velmax', 'rmax',
    'Age', 'Date', 'fname',
    'shapes1', 'calcul', 'large1',
    'split', 'split2', 'merge', 'merge2',
    'shapes2', 'interaction', 'interaction2',
    'area', 'deta'
]

df_ameda_renamed = df_ameda[[c for c in cols if c in df_ameda.columns]]
df_ameda_renamed.to_pickle('/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset/AMEDA/df_ameda_renamed.pkl')
df_ameda_renamed
